# N1 · POPE 幻觉探测 (POPE Hallucination)

> 配套 10.6-L2/L3 · 对不同幻觉率的模拟 VLM 跑 POPE, 看 yes-rate 偏离 0.5 量化幻觉,
> 加空白图诊断 (接 M10.3 模态坍缩)。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
FIG_SRC = Path.cwd().parent.parent / "research-figures" / "src"  # 复用 9.6 plotstyle
for p in (SRC, FIG_SRC): sys.path.insert(0, str(p))
import numpy as np, pandas as pd
import vlm_eval as ve
print('vlm_eval 就绪')

vlm_eval 就绪


## 1. POPE 探测集: 问「图里有 X 吗」, 正负各半 (L2)
真相一半「有」一半「没有」→ 一个诚实模型的 yes-rate 应 ≈ 0.5。

In [2]:
items = ve.make_probe_set(n=400, seed=1)
n_pos = sum(it['truly_present'] for it in items)
print(f"探测集: {len(items)} 题, {n_pos} 正 (有) / {len(items)-n_pos} 负 (没有) — 平衡")
print("理想 yes-rate = 0.5 (真相正负各半)")

探测集: 400 题, 200 正 (有) / 200 负 (没有) — 平衡
理想 yes-rate = 0.5 (真相正负各半)


## 2. 不同幻觉率 → POPE 指标 (幻觉率↑ → 准确率↓, yes率偏离 0.5)

In [3]:
rows = []
for h in [0.0, 0.2, 0.4, 0.6, 0.8]:
    r = ve.run_pope(hallucination=h, n=400, seed=1)
    rows.append({"幻觉率": h, "准确率": r["accuracy"], "yes率": r["yes_rate"],
                 "F1": r["f1"], "幻觉(FP)": r["confusion"]["fp_幻觉"]})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\n→ yes率越偏离 0.5, 模型越在'无中生有'。POPE 用 yes-bias 量化幻觉。")

 幻觉率   准确率  yes率    F1  幻觉(FP)
 0.0 0.938 0.438 0.933       0
 0.2 0.830 0.545 0.837      43
 0.4 0.705 0.670 0.748      93
 0.6 0.620 0.755 0.697     127
 0.8 0.525 0.850 0.648     165

→ yes率越偏离 0.5, 模型越在'无中生有'。POPE 用 yes-bias 量化幻觉。


## 3. 空白图诊断: 结合 M10.3 模态坍缩 (给「必然没有」的探测)

In [4]:
# 空白图诊断: 所有探测的物体都「不在」(模拟给空白图), 看模型 yes-rate
rng = np.random.default_rng(2)
blank_items = [{"object": f"o{i}", "truly_present": False} for i in range(200)]
for htag, h in [("低幻觉模型", 0.1), ("高幻觉模型", 0.6)]:
    ans = [ve.simulated_vlm_answer(it, hallucination=h, rng=rng) for it in blank_items]
    yes = np.mean(ans)
    print(f"{htag}: 对'必然没有'的探测, yes-rate = {yes:.2f}  ({'诚实说没有 ✅' if yes<0.2 else '大量幻觉 ⚠'})")
print("\n→ 给'图里啥都没有'的探测, 诚实模型几乎全说 no; 幻觉模型仍大量说 yes。")
print("→ 这是 POPE + 模态坍缩诊断 (M10.3-L4) 的结合: 戳穿'靠语言先验答题'。")

低幻觉模型: 对'必然没有'的探测, yes-rate = 0.11  (诚实说没有 ✅)
高幻觉模型: 对'必然没有'的探测, yes-rate = 0.60  (大量幻觉 ⚠)

→ 给'图里啥都没有'的探测, 诚实模型几乎全说 no; 幻觉模型仍大量说 yes。
→ 这是 POPE + 模态坍缩诊断 (M10.3-L4) 的结合: 戳穿'靠语言先验答题'。


## 4. 反思
你用 POPE + yes-bias 量化了视觉幻觉, 并用空白图诊断戳穿「靠语言先验答题」。带走:
- 幻觉 = 说图里没有的东西; 头号机理是语言先验过强 (L3, 接模态坍缩)。
- POPE 的精妙: 正负各半 + 诱饵负样本, 用 yes-rate 量化。
- 探测思想 = 设计「能戳穿偷懒」的题 (9.3 可证伪在 VLM 的体现)。
下一步 N2: 把评测结果画成诚实的出版级图 (复用 9.6)。